<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/SSL_MODEL_CREATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ECGFOUNDER — MODEL SETUP
# CELL 1: DOWNLOAD PRETRAINED 12-LEAD MODEL
# ============================================================

import os
import requests
from tqdm.auto import tqdm
from google.colab import drive
import requests
import os
from tqdm.auto import tqdm
drive.mount("/content/drive")

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/ECG_Project"

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "ECGFounder"
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "12_lead_ECGFounder.pth"
)

os.makedirs(MODEL_DIR, exist_ok=True)

# Official Hugging Face download URL
MODEL_URL = (
    "https://huggingface.co/PKUDigitalHealth/ECGFounder/"
    "resolve/main/12_lead_ECGFounder.pth"
)

# ------------------------------------------------------------
# DOWNLOAD
# ------------------------------------------------------------

# if os.path.exists(MODEL_PATH):

#     print("ECGFounder model already exists.")
#     print("Path:", MODEL_PATH)

# else:

#     print("Downloading ECGFounder...")
#     print("Size: approximately 370 MB")
#     print()

#     response = requests.get(
#         MODEL_URL,
#         stream=True
#     )

#     response.raise_for_status()

#     total_size = int(
#         response.headers.get("content-length", 0)
#     )

#     with open(MODEL_PATH, "wb") as f:

#         with tqdm(
#             total=total_size,
#             unit="B",
#             unit_scale=True,
#             desc="ECGFounder"
#         ) as pbar:

#             for chunk in response.iter_content(
#                 chunk_size=1024 * 1024
#             ):

#                 if chunk:

#                     f.write(chunk)
#                     pbar.update(len(chunk))

#     print("\nDownload complete.")

# # ------------------------------------------------------------
# # VERIFY FILE
# # ------------------------------------------------------------

# file_size_mb = (
#     os.path.getsize(MODEL_PATH)
#     / (1024 ** 2)
# )

# print("\n" + "=" * 60)
# print("ECGFOUNDER DOWNLOAD CHECK")
# print("=" * 60)

# print("File:")
# print(MODEL_PATH)

# print(f"\nSize: {file_size_mb:.2f} MB")

# print(
#     "Exists:",
#     os.path.exists(MODEL_PATH)
# )

# print("=" * 60)

Mounted at /content/drive


In [ ]:
# ============================================================
# ECGFOUNDER — CELL 2
# INSPECT PRETRAINED CHECKPOINT
# ============================================================

import torch
import os

MODEL_PATH = (
    "/content/drive/MyDrive/ECG_Project/"
    "ECGFounder/12_lead_ECGFounder.pth"
)

print("=" * 70)
print("LOADING ECGFOUNDER CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:")
print(type(checkpoint))

# ------------------------------------------------------------
# INSPECT TOP-LEVEL CONTENT
# ------------------------------------------------------------

if isinstance(checkpoint, dict):

    print("\nCheckpoint keys:")

    for key in checkpoint.keys():
        print(" -", key)

else:

    print("\nCheckpoint is not a dictionary.")

# ------------------------------------------------------------
# BASIC INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKPOINT INSPECTION COMPLETE")
print("=" * 70)

LOADING ECGFOUNDER CHECKPOINT

Checkpoint type:
<class 'dict'>

Checkpoint keys:
 - epoch
 - step
 - state_dict
 - optimizer
 - scheduler
 - val_auroc

CHECKPOINT INSPECTION COMPLETE


In [ ]:
# ============================================================
# ECGFOUNDER — CELL 3
# INSPECT MODEL WEIGHTS
# ============================================================

state_dict = checkpoint["state_dict"]

print("=" * 70)
print("ECGFOUNDER STATE DICTIONARY")
print("=" * 70)

print("\nNumber of parameter tensors:", len(state_dict))

print("\nFirst 30 parameter names:\n")

for i, (name, tensor) in enumerate(state_dict.items()):

    print(
        f"{i+1:2d}. "
        f"{name:<70} "
        f"shape={tuple(tensor.shape)}"
    )

    if i == 29:
        break



# ============================================================
# ECGFOUNDER — CELL 4
# INSPECT FINAL MODEL LAYERS
# ============================================================

print("=" * 70)
print("LAST 40 ECGFOUNDER PARAMETERS")
print("=" * 70)

items = list(state_dict.items())

for i, (name, tensor) in enumerate(items[-40:], start=len(items) - 39):

    print(
        f"{i:3d}. "
        f"{name:<70} "
        f"shape={tuple(tensor.shape)}"
    )


ECGFOUNDER STATE DICTIONARY

Number of parameter tensors: 509

First 30 parameter names:

 1. first_conv.conv.weight                                                 shape=(64, 12, 16)
 2. first_conv.conv.bias                                                   shape=(64,)
 3. first_bn.weight                                                        shape=(64,)
 4. first_bn.bias                                                          shape=(64,)
 5. first_bn.running_mean                                                  shape=(64,)
 6. first_bn.running_var                                                   shape=(64,)
 7. first_bn.num_batches_tracked                                           shape=()
 8. stage_list.0.block_list.0.bn1.weight                                   shape=(64,)
 9. stage_list.0.block_list.0.bn1.bias                                     shape=(64,)
10. stage_list.0.block_list.0.bn1.running_mean                             shape=(64,)
11. stage_list.0.block_list.0.bn1.ru

In [ ]:
import torch
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = "/content/drive/MyDrive/ECG_Project/ECGFounder/12_lead_ECGFounder.pth"

import numpy as np
from collections import Counter
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, data, label):
        self.data = data
        self.label = label

    def __getitem__(self, index):
        return (torch.tensor(self.data[index], dtype=torch.float), torch.tensor(self.label[index], dtype=torch.long))

    def __len__(self):
        return len(self.data)

class MyConv1dPadSame(nn.Module):
    """
    extend nn.Conv1d to support SAME padding

    input: (n_sample, in_channels, n_length)
    output: (n_sample, out_channels, (n_length+stride-1)//stride)
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups=1):
        super(MyConv1dPadSame, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups = groups
        self.conv = torch.nn.Conv1d(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            groups=self.groups)

    def forward(self, x):

        net = x

        # compute pad shape
        in_dim = net.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)
        # print(net.shape)
        net = self.conv(net)

        return net

class MyMaxPool1dPadSame(nn.Module):
    """
    extend nn.MaxPool1d to support SAME padding

    params:
        kernel_size: kernel size
        stride: the stride of the window. Default value is kernel_size

    input: (n_sample, n_channel, n_length)
    """
    def __init__(self, kernel_size):
        super(MyMaxPool1dPadSame, self).__init__()
        self.kernel_size = kernel_size
        self.max_pool = torch.nn.MaxPool1d(kernel_size=self.kernel_size)

    def forward(self, x):

        net = x

        # compute pad shape
        p = max(0, self.kernel_size - 1)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)

        net = self.max_pool(net)

        return net

class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class BasicBlock(nn.Module):
    """
    Basic Block:
        conv1 -> convk -> conv1

    params:
        in_channels: number of input channels
        out_channels: number of output channels
        ratio: ratio of channels to out_channels
        kernel_size: kernel window length
        stride: kernel step size
        groups: number of groups in convk
        downsample: whether downsample length
        use_bn: whether use batch_norm
        use_do: whether use dropout

    input: (n_sample, in_channels, n_length)
    output: (n_sample, out_channels, (n_length+stride-1)//stride)
    """
    def __init__(self, in_channels, out_channels, ratio, kernel_size, stride, groups, downsample, is_first_block=False, use_bn=True, use_do=True):
        super(BasicBlock, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.groups = groups
        self.downsample = downsample
        self.stride = stride if self.downsample else 1
        self.is_first_block = is_first_block
        self.use_bn = use_bn
        self.use_do = use_do

        self.middle_channels = int(self.out_channels * self.ratio)

        # the first conv, conv1
        self.bn1 = nn.BatchNorm1d(in_channels)
        self.activation1 = Swish()
        self.do1 = nn.Dropout(p=0.5)
        self.conv1 = MyConv1dPadSame(
            in_channels=self.in_channels,
            out_channels=self.middle_channels,
            kernel_size=1,
            stride=1,
            groups=1)

        # the second conv, convk
        self.bn2 = nn.BatchNorm1d(self.middle_channels)
        self.activation2 = Swish()
        self.do2 = nn.Dropout(p=0.5)
        self.conv2 = MyConv1dPadSame(
            in_channels=self.middle_channels,
            out_channels=self.middle_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            groups=self.groups)

        # the third conv, conv1
        self.bn3 = nn.BatchNorm1d(self.middle_channels)
        self.activation3 = Swish()
        self.do3 = nn.Dropout(p=0.5)
        self.conv3 = MyConv1dPadSame(
            in_channels=self.middle_channels,
            out_channels=self.out_channels,
            kernel_size=1,
            stride=1,
            groups=1)

        # Squeeze-and-Excitation
        r = 2
        self.se_fc1 = nn.Linear(self.out_channels, self.out_channels//r)
        self.se_fc2 = nn.Linear(self.out_channels//r, self.out_channels)
        self.se_activation = Swish()

        if self.downsample:
            self.max_pool = MyMaxPool1dPadSame(kernel_size=self.stride)

    def forward(self, x):

        identity = x

        out = x
        # the first conv, conv1
        if not self.is_first_block:
            if self.use_bn:
                out = self.bn1(out)
            out = self.activation1(out)
            if self.use_do:
                out = self.do1(out)
        out = self.conv1(out)

        # the second conv, convk
        if self.use_bn:
            out = self.bn2(out)
        out = self.activation2(out)
        if self.use_do:
            out = self.do2(out)
        out = self.conv2(out)

        # the third conv, conv1
        if self.use_bn:
            out = self.bn3(out)
        out = self.activation3(out)
        if self.use_do:
            out = self.do3(out)
        out = self.conv3(out) # (n_sample, n_channel, n_length)

        # Squeeze-and-Excitation
        se = out.mean(-1) # (n_sample, n_channel)
        se = self.se_fc1(se)
        se = self.se_activation(se)
        se = self.se_fc2(se)
        se = torch.sigmoid(se) # (n_sample, n_channel)
        out = torch.einsum('abc,ab->abc', out, se)

        # if downsample, also downsample identity
        if self.downsample:
            identity = self.max_pool(identity)

        # if expand channel, also pad zeros to identity
        if self.out_channels != self.in_channels:
            identity = identity.transpose(-1,-2)
            ch1 = (self.out_channels-self.in_channels)//2
            ch2 = self.out_channels-self.in_channels-ch1
            identity = F.pad(identity, (ch1, ch2), "constant", 0)
            identity = identity.transpose(-1,-2)

        # shortcut
        out += identity

        return out

class BasicStage(nn.Module):
    """
    Basic Stage:
        block_1 -> block_2 -> ... -> block_M
    """
    def __init__(self, in_channels, out_channels, ratio, kernel_size, stride, groups, i_stage, m_blocks, use_bn=True, use_do=True, verbose=False):
        super(BasicStage, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.groups = groups
        self.i_stage = i_stage
        self.m_blocks = m_blocks
        self.use_bn = use_bn
        self.use_do = use_do
        self.verbose = verbose

        self.block_list = nn.ModuleList()
        for i_block in range(self.m_blocks):

            # first block
            if self.i_stage == 0 and i_block == 0:
                self.is_first_block = True
            else:
                self.is_first_block = False
            # downsample, stride, input
            if i_block == 0:
                self.downsample = True
                self.stride = stride
                self.tmp_in_channels = self.in_channels
            else:
                self.downsample = False
                self.stride = 1
                self.tmp_in_channels = self.out_channels

            # build block
            tmp_block = BasicBlock(
                in_channels=self.tmp_in_channels,
                out_channels=self.out_channels,
                ratio=self.ratio,
                kernel_size=self.kernel_size,
                stride=self.stride,
                groups=self.groups,
                downsample=self.downsample,
                is_first_block=self.is_first_block,
                use_bn=self.use_bn,
                use_do=self.use_do)
            self.block_list.append(tmp_block)

    def forward(self, x):

        out = x

        for i_block in range(self.m_blocks):
            net = self.block_list[i_block]
            out = net(out)
            if self.verbose:
                print('stage: {}, block: {}, in_channels: {}, out_channels: {}, outshape: {}'.format(self.i_stage, i_block, net.in_channels, net.out_channels, list(out.shape)))
                print('stage: {}, block: {}, conv1: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv1.in_channels, net.conv1.out_channels, net.conv1.kernel_size, net.conv1.stride, net.conv1.groups))
                print('stage: {}, block: {}, convk: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv2.in_channels, net.conv2.out_channels, net.conv2.kernel_size, net.conv2.stride, net.conv2.groups))
                print('stage: {}, block: {}, conv1: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv3.in_channels, net.conv3.out_channels, net.conv3.kernel_size, net.conv3.stride, net.conv3.groups))

        return out

class Net1D(nn.Module):
    """

    Input:
        X: (n_samples, n_channel, n_length)
        Y: (n_samples)

    Output:
        out: (n_samples)

    params:
        in_channels
        base_filters
        filter_list: list, filters for each stage
        m_blocks_list: list, number of blocks of each stage
        kernel_size
        stride
        groups_width
        n_stages
        n_classes
        use_bn
        use_do

    """

    def __init__(self, in_channels, base_filters, ratio, filter_list, m_blocks_list, kernel_size, stride, groups_width, n_classes, use_bn=True, use_do=True, return_features=False, verbose=False):
        super(Net1D, self).__init__()

        self.in_channels = in_channels
        self.base_filters = base_filters
        self.ratio = ratio
        self.filter_list = filter_list
        self.m_blocks_list = m_blocks_list
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups_width = groups_width
        self.n_stages = len(filter_list)
        self.n_classes = n_classes
        self.use_bn = use_bn
        self.use_do = use_do
        self.return_features = return_features
        self.verbose = verbose

        # first conv
        self.first_conv = MyConv1dPadSame(
            in_channels=in_channels,
            out_channels=self.base_filters,
            kernel_size=self.kernel_size,
            stride=2)
        self.first_bn = nn.BatchNorm1d(base_filters)
        self.first_activation = Swish()

        # stages
        self.stage_list = nn.ModuleList()
        in_channels = self.base_filters
        for i_stage in range(self.n_stages):

            out_channels = self.filter_list[i_stage]
            m_blocks = self.m_blocks_list[i_stage]
            tmp_stage = BasicStage(
                in_channels=in_channels,
                out_channels=out_channels,
                ratio=self.ratio,
                kernel_size=self.kernel_size,
                stride=self.stride,
                groups=out_channels//self.groups_width,
                i_stage=i_stage,
                m_blocks=m_blocks,
                use_bn=self.use_bn,
                use_do=self.use_do,
                verbose=self.verbose)
            self.stage_list.append(tmp_stage)
            in_channels = out_channels

        # final prediction
        self.dense = nn.Linear(in_channels, n_classes)

    def forward(self, x):

        out = x

        # first conv
        out = self.first_conv(out)
        if self.use_bn:
            out = self.first_bn(out)
        out = self.first_activation(out)

        # stages
        for i_stage in range(self.n_stages):
            net = self.stage_list[i_stage]
            out = net(out)

        # final prediction
        deep_features = out.mean(-1)
        out = self.dense(deep_features)

        if self.return_features:
            return out, deep_features
        else:
            return out

if __name__ == "__main__":
  model = Net1D(
      in_channels=12,
      base_filters=64, #32 64
      ratio=1,
      filter_list=[64,160,160,400,400,1024,1024],    #[16,32,32,80,80,256,256] [32,64,64,160,160,512,512] [64,160,160,400,400,1024,1024]
      m_blocks_list=[2,2,2,3,3,4,4],   #[2,2,2,2,2,2,2] [2,2,2,3,3,4,4]
      kernel_size=16,
      stride=2,
      groups_width=16,
      verbose=True,
      use_bn=False,
      use_do=False,
      n_classes=150)
  print(model)



Net1D(
  (first_conv): MyConv1dPadSame(
    (conv): Conv1d(12, 64, kernel_size=(16,), stride=(2,))
  )
  (first_bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (first_activation): Swish()
  (stage_list): ModuleList(
    (0): BasicStage(
      (block_list): ModuleList(
        (0): BasicBlock(
          (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation1): Swish()
          (do1): Dropout(p=0.5, inplace=False)
          (conv1): MyConv1dPadSame(
            (conv): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
          )
          (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation2): Swish()
          (do2): Dropout(p=0.5, inplace=False)
          (conv2): MyConv1dPadSame(
            (conv): Conv1d(64, 64, kernel_size=(16,), stride=(2,), groups=4)
          )
          (bn3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [ ]:
# Build exact ECGFounder architecture
model = Net1D(
    in_channels=12,
    base_filters=64,
    ratio=1,
    filter_list=[64, 160, 160, 400, 400, 1024, 1024],
    m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
    kernel_size=16,
    stride=2,
    groups_width=16,
    n_classes=150,
    use_bn=False,
    use_do=False,
    return_features=True,
    verbose=False
)

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["state_dict"])

model = model.to(DEVICE)

print("✓ ECGFounder loaded successfully")
print("Device:", DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

✓ ECGFounder loaded successfully
Device: cuda
Parameters: 30819014


In [ ]:
# ============================================================
# CELL 5 — UNLABELED ACS DATASET FOR DOMAIN ADAPTATION
# ============================================================

import h5py
import torch
from torch.utils.data import Dataset, DataLoader


TRAIN_H5 = (
    "/content/drive/MyDrive/ECG_Project/"
    "results/preprocessing/processed/train_fixed.h5"
)


class ACSSelfSupervisedDataset(Dataset):

    def __init__(self, h5_path):
        self.h5_path = h5_path

        with h5py.File(self.h5_path, "r") as f:
            self.length = len(f["signals"])

        self.h5 = None

        print("ACS training ECGs:", self.length)


    def __len__(self):
        return self.length


    def __getitem__(self, idx):

        if self.h5 is None:
            self.h5 = h5py.File(
                self.h5_path,
                "r"
            )

        ecg = self.h5["signals"][idx]

        ecg = torch.tensor(
            ecg,
            dtype=torch.float32
        )

        return ecg


train_ssl_dataset = ACSSelfSupervisedDataset(
    TRAIN_H5
)

train_ssl_loader = DataLoader(
    train_ssl_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

print("Batches:", len(train_ssl_loader))

ACS training ECGs: 15273
Batches: 478


In [ ]:
x = next(iter(train_ssl_loader))

print("Batch shape:", x.shape)
print("dtype:", x.dtype)
print("min:", x.min().item())
print("max:", x.max().item())
print("mean:", x.mean().item())
print("std:", x.std().item())

Batch shape: torch.Size([32, 12, 5000])
dtype: torch.float32
min: -23.415489196777344
max: 23.00204849243164
mean: -3.178914423362933e-11
std: 1.000000238418579


In [ ]:
# ============================================================
# CELL 6 — SAFE ECG AUGMENTATIONS
# ============================================================

import torch
import random


class SafeECGAugment:

    def __init__(
        self,
        noise_std=0.01,
        scale_range=(0.95, 1.05),
        max_mask_samples=25,
        p_noise=0.5,
        p_scale=0.5,
        p_mask=0.3
    ):
        self.noise_std = noise_std
        self.scale_range = scale_range
        self.max_mask_samples = max_mask_samples

        self.p_noise = p_noise
        self.p_scale = p_scale
        self.p_mask = p_mask


    def __call__(self, x):

        x = x.clone()

        # --------------------------------
        # 1. Tiny Gaussian noise
        # --------------------------------
        if random.random() < self.p_noise:

            noise = torch.randn_like(x) * self.noise_std

            x = x + noise


        # --------------------------------
        # 2. Mild amplitude scaling
        # --------------------------------
        if random.random() < self.p_scale:

            scale = random.uniform(
                self.scale_range[0],
                self.scale_range[1]
            )

            x = x * scale


        # --------------------------------
        # 3. Small temporal masking
        # --------------------------------
        if random.random() < self.p_mask:

            mask_len = random.randint(
                1,
                self.max_mask_samples
            )

            start = random.randint(
                0,
                x.shape[-1] - mask_len
            )

            x[:, start:start + mask_len] = 0


        return x

In [ ]:
# ============================================================
# CELL 7 — TWO-VIEW SELF-SUPERVISED DATASET
# ============================================================

augment = SafeECGAugment()


class ACSContrastiveDataset(Dataset):

    def __init__(
        self,
        h5_path,
        augment
    ):

        self.h5_path = h5_path
        self.augment = augment

        with h5py.File(
            self.h5_path,
            "r"
        ) as f:

            self.length = len(
                f["signals"]
            )

        self.h5 = None

        print(
            "ACS contrastive ECGs:",
            self.length
        )


    def __len__(self):
        return self.length


    def __getitem__(self, idx):

        if self.h5 is None:

            self.h5 = h5py.File(
                self.h5_path,
                "r"
            )

        ecg = torch.tensor(
            self.h5["signals"][idx],
            dtype=torch.float32
        )

        view1 = self.augment(ecg)
        view2 = self.augment(ecg)

        return view1, view2

In [ ]:
train_ssl_dataset = ACSContrastiveDataset(
    TRAIN_H5,
    augment
)

train_ssl_loader = DataLoader(
    train_ssl_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

ACS contrastive ECGs: 15273


In [ ]:
v1, v2 = next(
    iter(train_ssl_loader)
)

print("View 1:", v1.shape)
print("View 2:", v2.shape)

print(
    "Mean absolute difference:",
    torch.mean(
        torch.abs(v1 - v2)
    ).item()
)


View 1: torch.Size([32, 12, 5000])
View 2: torch.Size([32, 12, 5000])
Mean absolute difference: 0.016484459862113


In [ ]:
# ============================================================
# CELL 8 — CONTRASTIVE PROJECTION HEAD
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class ProjectionHead(nn.Module):

    def __init__(
        self,
        input_dim=1024,
        hidden_dim=256,
        output_dim=128
    ):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                output_dim
            )
        )


    def forward(self, x):

        z = self.net(x)

        # Important for contrastive learning
        z = F.normalize(
            z,
            dim=1
        )

        return z


projection_head = ProjectionHead(
    input_dim=1024,
    hidden_dim=256,
    output_dim=128
).to(DEVICE)

print(projection_head)

ProjectionHead(
  (net): Sequential(
    (0): Linear(in_features=1024, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
  )
)


In [ ]:
# ============================================================
# CELL 9 — PREPARE ECGFOUNDER FOR SELF-SUPERVISED ADAPTATION
# ============================================================

model.return_features = True

# Freeze the old 150-class classifier
for param in model.dense.parameters():
    param.requires_grad = False

print("ECGFounder will now return deep features.")

ECGFounder will now return deep features.


In [ ]:
# ============================================================
# CELL 10 — SELECT ECGFOUNDER LAYERS TO ADAPT
# ============================================================

SSL_UNFREEZE_MODE = "last2"


def configure_ssl_backbone(
    model,
    mode="last2"
):

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Keep original classifier frozen
    for param in model.dense.parameters():
        param.requires_grad = False


    if mode == "last1":

        for p in model.stage_list[-1].parameters():
            p.requires_grad = True


    elif mode == "last2":

        for stage in model.stage_list[-2:]:

            for p in stage.parameters():
                p.requires_grad = True


    elif mode == "last3":

        for stage in model.stage_list[-3:]:

            for p in stage.parameters():
                p.requires_grad = True


    elif mode == "full":

        for p in model.parameters():
            p.requires_grad = True

        # Still do NOT train old 150-class head
        for p in model.dense.parameters():
            p.requires_grad = False


    else:

        raise ValueError(
            "Choose: last1, last2, last3 or full"
        )


    return model


model = configure_ssl_backbone(
    model,
    SSL_UNFREEZE_MODE
)


trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print("SSL mode:", SSL_UNFREEZE_MODE)

print(
    f"Trainable ECGFounder parameters: "
    f"{trainable:,} / {total:,}"
)

print(
    f"Trainable percentage: "
    f"{100*trainable/total:.2f}%"
)

SSL mode: last2
Trainable ECGFounder parameters: 26,708,768 / 30,819,014
Trainable percentage: 86.66%


In [ ]:
# ============================================================
# CELL 11 — NT-XENT CONTRASTIVE LOSS
# ============================================================

def nt_xent_loss(
    z1,
    z2,
    temperature=0.1
):

    batch_size = z1.size(0)

    # Combine both views
    z = torch.cat(
        [z1, z2],
        dim=0
    )

    # Already normalized by projection head
    similarity = torch.matmul(
        z,
        z.T
    )

    similarity = (
        similarity / temperature
    )

    # Remove self-similarity
    mask = torch.eye(
        2 * batch_size,
        device=z.device,
        dtype=torch.bool
    )

    similarity = similarity.masked_fill(
        mask,
        -1e9
    )

    # Positive pair mapping:
    #
    # 0      ↔ batch_size
    # 1      ↔ batch_size + 1
    # ...
    #
    targets = torch.arange(
        2 * batch_size,
        device=z.device
    )

    targets = (
        targets + batch_size
    ) % (2 * batch_size)

    loss = F.cross_entropy(
        similarity,
        targets
    )

    return loss

In [ ]:
# ============================================================
# CELL 12 — SSL OPTIMIZER
# ============================================================

backbone_params = [
    p
    for p in model.parameters()
    if p.requires_grad
]

optimizer_ssl = torch.optim.AdamW(
    [
        {
            "params": backbone_params,
            "lr": 3e-6
        },

        {
            "params": projection_head.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

print("SSL optimizer ready.")

SSL optimizer ready.


In [ ]:
# ============================================================
# CELL 13 — ACS DOMAIN ADAPTATION TRAINING
# ============================================================

import os
import numpy as np
from tqdm.auto import tqdm


SSL_EPOCHS = 5

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "ECG_Project/Domain_Adaptation"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


for epoch in range(
    1,
    SSL_EPOCHS + 1
):

    model.train()
    projection_head.train()

    epoch_losses = []


    progress = tqdm(
        train_ssl_loader,
        desc=f"SSL Epoch {epoch}/{SSL_EPOCHS}"
    )


    for view1, view2 in progress:

        view1 = view1.to(
            DEVICE,
            non_blocking=True
        )

        view2 = view2.to(
            DEVICE,
            non_blocking=True
        )


        optimizer_ssl.zero_grad()


        # --------------------------------
        # ECGFounder representations
        # --------------------------------

        _, features1 = model(view1)
        _, features2 = model(view2)


        # Expected:
        # [batch, 1024]

        # --------------------------------
        # Projection
        # --------------------------------

        z1 = projection_head(
            features1
        )

        z2 = projection_head(
            features2
        )


        # --------------------------------
        # Contrastive loss
        # --------------------------------

        loss = nt_xent_loss(
            z1,
            z2,
            temperature=0.1
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            backbone_params,
            max_norm=5.0
        )


        optimizer_ssl.step()


        epoch_losses.append(
            loss.item()
        )


        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    avg_loss = np.mean(
        epoch_losses
    )


    print(
        f"\nEpoch {epoch}: "
        f"Contrastive Loss = "
        f"{avg_loss:.5f}"
    )


    # --------------------------------
    # Save every adapted epoch
    # --------------------------------

    save_path = os.path.join(
        SAVE_DIR,
        f"ecgfounder_acs_ssl_epoch{epoch}.pth"
    )


    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "projection_head_state_dict":
                projection_head.state_dict(),

            "ssl_mode":
                SSL_UNFREEZE_MODE,

            "contrastive_loss":
                avg_loss
        },

        save_path
    )


    print(
        "Saved:",
        save_path
    )

SSL Epoch 1/5:   0%|          | 0/478 [00:00<?, ?it/s]


Epoch 1: Contrastive Loss = 0.02570
Saved: /content/drive/MyDrive/ECG_Project/Domain_Adaptation/ecgfounder_acs_ssl_epoch1.pth


SSL Epoch 2/5:   0%|          | 0/478 [00:00<?, ?it/s]


Epoch 2: Contrastive Loss = 0.00791
Saved: /content/drive/MyDrive/ECG_Project/Domain_Adaptation/ecgfounder_acs_ssl_epoch2.pth


SSL Epoch 3/5:   0%|          | 0/478 [00:00<?, ?it/s]


Epoch 3: Contrastive Loss = 0.00702
Saved: /content/drive/MyDrive/ECG_Project/Domain_Adaptation/ecgfounder_acs_ssl_epoch3.pth


SSL Epoch 4/5:   0%|          | 0/478 [00:00<?, ?it/s]


Epoch 4: Contrastive Loss = 0.00651
Saved: /content/drive/MyDrive/ECG_Project/Domain_Adaptation/ecgfounder_acs_ssl_epoch4.pth


SSL Epoch 5/5:   0%|          | 0/478 [00:00<?, ?it/s]


Epoch 5: Contrastive Loss = 0.00637
Saved: /content/drive/MyDrive/ECG_Project/Domain_Adaptation/ecgfounder_acs_ssl_epoch5.pth
